In [1]:
import os
import random
import numpy as np
import pandas as pd
import imageio

# Reproducibility (score-stable; does not change core logic)
SEED = 1337
random.seed(SEED)
np.random.seed(SEED)

# Robust base path for this dataset in Kaggle filesystem
BASE = "/kaggle/input/aerial-cactus-identification"
TRAIN_CSV = os.path.join(BASE, "train.csv")
TRAIN_DIR = os.path.join(BASE, "train")
TEST_DIR = os.path.join(BASE, "test")
SAMPLE_SUB = os.path.join(BASE, "sample_submission.csv")

assert os.path.exists(TRAIN_CSV), f"Missing train.csv at {TRAIN_CSV}"
assert os.path.isdir(TRAIN_DIR), f"Missing train dir at {TRAIN_DIR}"
assert os.path.isdir(TEST_DIR), f"Missing test dir at {TEST_DIR}"
assert os.path.exists(SAMPLE_SUB), f"Missing sample_submission.csv at {SAMPLE_SUB}"



In [2]:
df = pd.read_csv(TRAIN_CSV)
df.head()




,id,has_cactus
0,2de8f189f1dce439766637e75df0ee27.jpg,1
1,36704d250f236238e7f996812c48235d.jpg,1
2,eacde22fdc8c175972a5768e3daa8bc9.jpg,1
3,5d442f834da5e57d22b24802c32a8ca8.jpg,1
4,152491e0daf75c0e669400300ff7e645.jpg,1


In [3]:
def load_images(df_ids: pd.DataFrame, folder: str):
    # Fix: ensure dtype float32 for Keras/TF speed/memory; keep same normalization semantics
    images = np.zeros((len(df_ids), 32, 32, 3), dtype=np.float32)

    for i, file in enumerate(df_ids.id):
        images[i] = imageio.imread(os.path.join(folder, file))

    # Keep original scaling: (x - 128) / 64
    return (images - 128.0) / 64.0


images = load_images(df, TRAIN_DIR)
images.shape, images.dtype



/tmp/ipykernel_11/2427837086.py:6: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  images[i] = imageio.imread(os.path.join(folder, file))


((14175, 32, 32, 3), dtype('float32'))

In [4]:
# Fix: keras==3 in this environment can crash due to protobuf incompat; use tf_keras instead.
from tf_keras.models import Model
from tf_keras.layers import MaxPooling2D, GlobalAveragePooling2D
from tf_keras.layers import Input, ReLU, BatchNormalization, Conv2D, Activation


def ConvCell(m, filters, kernel=3):
    m = Conv2D(filters, kernel, padding="same")(m)
    m = BatchNormalization()(m)
    m = ReLU()(m)
    return m


def DeepConvCell(m, n, filters, kernel=3):
    for _ in range(n):
        m = ConvCell(m, filters, kernel)
    m = MaxPooling2D()(m)
    return m


n_inp = Input(shape=(32, 32, 3))
conv0 = DeepConvCell(n_inp, 3, 32)
conv1 = DeepConvCell(conv0, 2, 64)
conv2 = DeepConvCell(conv1, 1, 128)

convF = Conv2D(1, 3, padding="same")(conv2)
gl_avg_pool = GlobalAveragePooling2D()(convF)
fc = Activation("sigmoid")(gl_avg_pool)

m = Model(inputs=n_inp, outputs=fc)
m.compile(loss="binary_crossentropy", optimizer="adam")
m.summary()



2026-01-10 20:18:44.931876: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768076324.944995      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768076324.949006      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-10 20:18:44.965101: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 32, 32, 3)]       0         
                                                                 
 conv2d (Conv2D)             (None, 32, 32, 32)        896       
                                                                 
 batch_normalization (Batch  (None, 32, 32, 32)        128       
 Normalization)                                                  
                                                                 
 re_lu (ReLU)                (None, 32, 32, 32)        0         
                                                                 
 conv2d_1 (Conv2D)           (None, 32, 32, 32)        9248      
                                                                 
 batch_normalization_1 (Bat  (None, 32, 32, 32)        128       
 chNormalization)                                            

2026-01-10 20:18:51.696151: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1768076331.696594      11 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:c5:00.0, compute capability: 8.6


In [5]:
# Fix: imbalanced-learn import fails due to sklearn mismatch in environment.
# Minimal replacement: random oversampling to balance classes (same intent as RandomOverSampler).
y = df.has_cactus.values.astype(np.uint8)

pos_idx = np.where(y == 1)[0]
neg_idx = np.where(y == 0)[0]

# Oversample minority to match majority
if len(pos_idx) < len(neg_idx):
    minority, majority = pos_idx, neg_idx
else:
    minority, majority = neg_idx, pos_idx

n_to_add = len(majority) - len(minority)
if n_to_add > 0:
    add_idx = np.random.choice(minority, size=n_to_add, replace=True)
    idx = np.concatenate([np.arange(len(y)), add_idx])
else:
    idx = np.arange(len(y))

data = images[idx]
target = y[idx]

# Shuffle after resample
perm = np.random.permutation(len(target))
data = data[perm]
target = target[perm]

data.shape, target.mean(), len(target)



((21256, 32, 32, 3), 0.5, 21256)

In [6]:
# Training: unchanged core training call; just now data/target exists.
s = m.fit(data, target, batch_size=64, epochs=50, validation_split=0.33, verbose=2)



Epoch 1/50


I0000 00:00:1768076333.330619      65 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1768076334.051351      68 service.cc:148] XLA service 0x7f35d9030930 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1768076334.051373      68 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-01-10 20:18:54.056445: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1768076334.117952      68 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


223/223 - 5s - loss: 0.0993 - val_loss: 0.2962 - 5s/epoch - 21ms/step
Epoch 2/50
223/223 - 1s - loss: 0.0543 - val_loss: 0.0339 - 894ms/epoch - 4ms/step
Epoch 3/50
223/223 - 1s - loss: 0.0331 - val_loss: 0.0709 - 902ms/epoch - 4ms/step
Epoch 4/50
223/223 - 1s - loss: 0.0314 - val_loss: 0.0223 - 894ms/epoch - 4ms/step
Epoch 5/50
223/223 - 1s - loss: 0.0133 - val_loss: 0.0143 - 896ms/epoch - 4ms/step
Epoch 6/50
223/223 - 1s - loss: 0.0127 - val_loss: 0.0080 - 908ms/epoch - 4ms/step
Epoch 7/50
223/223 - 1s - loss: 0.0084 - val_loss: 0.0228 - 902ms/epoch - 4ms/step
Epoch 8/50
223/223 - 1s - loss: 0.0077 - val_loss: 0.0076 - 898ms/epoch - 4ms/step
Epoch 9/50
223/223 - 1s - loss: 0.0050 - val_loss: 0.0255 - 897ms/epoch - 4ms/step
Epoch 10/50
223/223 - 1s - loss: 0.0052 - val_loss: 0.0108 - 899ms/epoch - 4ms/step
Epoch 11/50
223/223 - 1s - loss: 0.0027 - val_loss: 0.0054 - 891ms/epoch - 4ms/step
Epoch 12/50
223/223 - 1s - loss: 4.9114e-04 - val_loss: 0.0052 - 897ms/epoch - 4ms/step
Epoch 13/5

In [7]:
# Fix: build test dataframe with correct length, and preserve id ordering as in sample_submission.
sub = pd.read_csv(SAMPLE_SUB)
test_ids = sub["id"].tolist()

# Sanity check: all files exist
missing = [f for f in test_ids if not os.path.exists(os.path.join(TEST_DIR, f))]
assert len(missing) == 0, f"Missing {len(missing)} test files, e.g. {missing[:3]}"

df2 = pd.DataFrame(
    {"id": test_ids, "has_cactus": np.zeros(len(test_ids), dtype=np.float32)}
)

# Predict probabilities (do NOT threshold for ROC AUC submission)
p = (
    m.predict(load_images(df2, TEST_DIR), batch_size=256, verbose=0)
    .reshape(-1)
    .astype(np.float32)
)

df2["has_cactus"] = np.clip(p, 0.0, 1.0)
df2.head()



/tmp/ipykernel_11/2427837086.py:6: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  images[i] = imageio.imread(os.path.join(folder, file))


,id,has_cactus
0,09034a34de0e2015a8a28dfe18f423f6.jpg,1.0
1,134f04305c795d6d202502c2ce3578f3.jpg,1.0
2,41fad8d145e6c41868ce3617e30a2545.jpg,1.0
3,35f8a11352c8d41b6231bb33d8d09f7e.jpg,1.0
4,b77dc902b035887cbbc01920ce0e3151.jpg,1.0


In [8]:
# Write valid Kaggle submission
df2.to_csv("submission.csv", index=False)
print("Wrote submission.csv with shape:", df2.shape)
print(df2.head())

Wrote submission.csv with shape: (3325, 2)
                                     id  has_cactus
0  09034a34de0e2015a8a28dfe18f423f6.jpg         1.0
1  134f04305c795d6d202502c2ce3578f3.jpg         1.0
2  41fad8d145e6c41868ce3617e30a2545.jpg         1.0
3  35f8a11352c8d41b6231bb33d8d09f7e.jpg         1.0
4  b77dc902b035887cbbc01920ce0e3151.jpg         1.0
